In [1]:
import os
import lmetk
print(os.environ['PROJECT_ROOT'])  # should print the detected root

import hydra
from lmetk.data.utils import reconstruct_datetime_from_features
import numpy as np
from lmetk.data.datasets import SWDataset, SWBatchDataset

%load_ext autoreload
%autoreload 2

/home/levtel/projects/TabPHM


### Refactored version

In [2]:
data_name = "ETTh1"

with hydra.initialize(version_base="1.3", config_path="../configs", job_name="run"):
    # print('Current config file: ', config_file)
    parameters = hydra.compose(
        config_name="run.yaml",
        overrides=[f"datasource={data_name}"],
        return_hydra_config=True,
    )
    dataset_loader2 = hydra.utils.instantiate(parameters.datasource)
    dataset_loader2._pipeline()


# Get params of splitting and sliding
data_splitter_params = parameters.datasource.data_splitter
seq_len = data_splitter_params.seq_len
label_len = data_splitter_params.label_len
pred_len = data_splitter_params.pred_len

# get data
splitted_data, splitted_timestamps = dataset_loader2.get_data()

dataset_train = SWDataset(
    features=splitted_data["train"],
    time_features=splitted_timestamps["train"],
    seq_len=seq_len,
    label_len=label_len,
    pred_len=pred_len,
)

dataset_val = SWDataset(
    features=splitted_data["val"],
    time_features=splitted_timestamps["val"],
    seq_len=seq_len,
    label_len=label_len,
    pred_len=pred_len,
)
dataset_test = SWDataset(
    features=splitted_data["test"],
    time_features=splitted_timestamps["test"],
    seq_len=seq_len,
    label_len=label_len,
    pred_len=pred_len,
)
print(f"Train dataset size: {dataset_train.__len__()}")
print(f"Validation dataset size: {dataset_val.__len__()}")
print(f"Test dataset size: {dataset_test.__len__()}")

Train dataset size: 8449
Validation dataset size: 2785
Test dataset size: 5805


In [7]:
for i in range(2783, 2787):
    seq_x, seq_y, seq_x_mark, seq_y_mark = dataset_val.__getitem__(i)
    print("Index", i)
    print(
        f"seq_x shape: {seq_x.shape}, seq_y shape: {seq_y.shape}, seq_x_mark shape: {seq_x_mark.shape}, seq_y_mark shape: {seq_y_mark.shape}"
    )

Index 2783
seq_x shape: torch.Size([96, 1]), seq_y shape: torch.Size([144, 1]), seq_x_mark shape: torch.Size([96, 4]), seq_y_mark shape: torch.Size([144, 4])
Index 2784
seq_x shape: torch.Size([96, 1]), seq_y shape: torch.Size([144, 1]), seq_x_mark shape: torch.Size([96, 4]), seq_y_mark shape: torch.Size([144, 4])


AssertionError: Target sequence indices out of bounds

In [8]:
for mode in ["val"]:
    batch_dataset = SWBatchDataset(
        features=splitted_data[mode],
        time_features=splitted_timestamps[mode],
        seq_len=seq_len,
        label_len=label_len,
        pred_len=pred_len,
        stride=1,
    )
    print(f" {mode} size: {batch_dataset.__len__()}")
    for i in range(batch_dataset.__len__()):
        batch_dataset.__getitem__(i)
    seq_x2, seq_y2, seq_x_mark2, seq_y_mark2 = batch_dataset.__getitem__(i)


seq_x2, seq_y2, seq_x_mark2, seq_y_mark2 = batch_dataset.__getitem__(4)
seq_x1, seq_y1, seq_x_mark1, seq_y_mark1 = dataset_val.__getitem__(4)


np.testing.assert_array_almost_equal(seq_x2, seq_x1)

 val size: 2785


### Railway: classical dataset 

In [9]:
import hydra

data_name = "railway"

with hydra.initialize(version_base="1.3", config_path="../configs", job_name="run"):
    # print('Current config file: ', config_file)
    parameters = hydra.compose(
        config_name="run.yaml",
        overrides=[f"datasource={data_name}"],
        return_hydra_config=True,
    )
    dataset_loader2 = hydra.utils.instantiate(parameters.datasource)
    dataset_loader2._pipeline()

In [10]:
data_splitter_params = parameters.datasource.data_splitter
seq_len = data_splitter_params.seq_len
label_len = data_splitter_params.label_len
pred_len = data_splitter_params.pred_len

# get data
splitted_data, splitted_timestamps = dataset_loader2.get_data()

dataset_train = SWDataset(
    features=splitted_data["train"],
    time_features=splitted_timestamps["train"],
    seq_len=seq_len,
    label_len=label_len,
    pred_len=pred_len,
)

dataset_val = SWDataset(
    features=splitted_data["val"],
    time_features=splitted_timestamps["val"],
    seq_len=seq_len,
    label_len=label_len,
    pred_len=pred_len,
)
dataset_test = SWDataset(
    features=splitted_data["test"],
    time_features=splitted_timestamps["test"],
    seq_len=seq_len,
    label_len=label_len,
    pred_len=pred_len,
)
print(f"Train dataset size: {dataset_train.__len__()}")
print(f"Validation dataset size: {dataset_val.__len__()}")
print(f"Test dataset size: {dataset_test.__len__()}")

Train dataset size: 37921
Validation dataset size: 4201
Test dataset size: 8185


In [11]:
# Test should stard from "2023-01-24 00:00:00" (if apply_lookback == False then __getitem__(seq_len) otherwise __getitem__(0))
seq_x, seq_y, seq_x_mark, seq_y_mark = dataset_test.__getitem__(seq_len)
reconstruct_datetime_from_features(
    seq_x_mark.numpy(), ["year", "month", "day", "weekday", "hour"]
)[:3]

0   2023-01-24 00:00:00
1   2023-01-24 01:00:00
2   2023-01-24 02:00:00
dtype: datetime64[ns]

In [12]:
str(
    reconstruct_datetime_from_features(
        seq_x_mark.numpy(), ["year", "month", "day", "weekday", "hour"]
    )[0]
) == "2023-01-24 00:00:00"

True

In [13]:
seq_x, seq_y, seq_x_mark, seq_y_mark = dataset_val.__getitem__(seq_len)
reconstruct_datetime_from_features(
    seq_x_mark.numpy(), ["year", "month", "day", "weekday", "hour"]
)[:3]

0   2022-08-02 00:00:00
1   2022-08-02 01:00:00
2   2022-08-02 02:00:00
dtype: datetime64[ns]

### Check of tsfresh

In [49]:
import pandas as pd

df = dataset_loader2.df
df_stamp = dataset_loader2.df_stamp

df = df.rename(columns={'Bahnlast':'y'})

df_stamp = df_stamp.to_frame()
df_stamp = df_stamp.rename(columns={'datetime':'time'})

new_df = pd.concat([df_stamp, df], axis=1)

new_df['id'] = pd.Series(['A'] * new_df.shape[0])

,id,time,y,Pully
0,A,2018-04-02 00:00:00,130.204375,6.300
1,A,2018-04-02 01:00:00,87.320175,6.350
2,A,2018-04-02 02:00:00,79.965675,5.525
3,A,2018-04-02 03:00:00,84.284475,4.900
4,A,2018-04-02 04:00:00,108.719675,5.450
...,...,...,...,...
95,A,2018-04-05 23:00:00,252.608675,8.400
96,A,2018-04-06 00:00:00,207.917800,8.200
97,A,2018-04-06 01:00:00,143.897250,7.925
98,A,2018-04-06 02:00:00,142.065100,6.950


In [140]:
from tsfresh.utilities.dataframe_functions import roll_time_series
short_df = new_df[['id','time', 'y', 'Pully']].iloc[:49]
df_rolled = roll_time_series(short_df, column_id="id", column_sort="time", rolling_direction=24, max_timeshift=24, min_timeshift=24)

df_rolled



Rolling: 100%|██████████| 3/3 [00:00<00:00, 444.55it/s]


,id,time,y,Pully
25,"(A, 2018-04-03 00:00:00)",2018-04-02 00:00:00,130.204375,6.300
26,"(A, 2018-04-03 00:00:00)",2018-04-02 01:00:00,87.320175,6.350
27,"(A, 2018-04-03 00:00:00)",2018-04-02 02:00:00,79.965675,5.525
28,"(A, 2018-04-03 00:00:00)",2018-04-02 03:00:00,84.284475,4.900
29,"(A, 2018-04-03 00:00:00)",2018-04-02 04:00:00,108.719675,5.450
30,"(A, 2018-04-03 00:00:00)",2018-04-02 05:00:00,174.614275,5.350
31,"(A, 2018-04-03 00:00:00)",2018-04-02 06:00:00,254.527350,5.575
32,"(A, 2018-04-03 00:00:00)",2018-04-02 07:00:00,264.813075,5.600
33,"(A, 2018-04-03 00:00:00)",2018-04-02 08:00:00,271.176775,5.075
34,"(A, 2018-04-03 00:00:00)",2018-04-02 09:00:00,252.122900,5.650


In [138]:
short_df

,id,time,y,Pully
0,A,2018-04-02 00:00:00,130.204375,6.300
1,A,2018-04-02 01:00:00,87.320175,6.350
2,A,2018-04-02 02:00:00,79.965675,5.525
3,A,2018-04-02 03:00:00,84.284475,4.900
4,A,2018-04-02 04:00:00,108.719675,5.450
5,A,2018-04-02 05:00:00,174.614275,5.350
6,A,2018-04-02 06:00:00,254.527350,5.575
7,A,2018-04-02 07:00:00,264.813075,5.600
8,A,2018-04-02 08:00:00,271.176775,5.075
9,A,2018-04-02 09:00:00,252.122900,5.650


In [156]:
df = pd.DataFrame({
   "id": [1, 1, 1, 1, 2, 2],
   "time": [1, 2, 3, 4, 8, 9],
   "x": [1, 2, 3, 4, 10, 11],
   "y": [5, 6, 7, 8, 12, 13],
})
df

,id,time,x,y
0,1,1,1,5
1,1,2,2,6
2,1,3,3,7
3,1,4,4,8
4,2,8,10,12
5,2,9,11,13


In [162]:

df_rolled = roll_time_series(df, column_id="id", column_sort="time", rolling_direction=1, max_timeshift=1, min_timeshift=0)
df_rolled

Rolling: 100%|██████████| 4/4 [00:00<00:00, 536.10it/s]


,id,time,x,y
2,"(1, 1)",1,1,5
6,"(1, 2)",1,1,5
7,"(1, 2)",2,2,6
0,"(1, 3)",2,2,6
1,"(1, 3)",3,3,7
4,"(1, 4)",3,3,7
5,"(1, 4)",4,4,8
3,"(2, 8)",8,10,12
8,"(2, 9)",8,10,12
9,"(2, 9)",9,11,13


In [163]:
from tsfresh import extract_features
df_features = extract_features(df_rolled, column_id="id", column_sort="time")
df_features['x__sum_values']

Feature Extraction: 100%|██████████| 12/12 [00:00<00:00, 236.55it/s]


1  1     1.0
   2     3.0
   3     5.0
   4     7.0
2  8    10.0
   9    21.0
Name: x__sum_values, dtype: float64

In [159]:
from tsfresh.utilities.dataframe_functions import make_forecasting_frame
x,y = make_forecasting_frame(df['x'], max_timeshift=1, rolling_direction=1, kind='kind')

Rolling: 100%|██████████| 6/6 [00:00<00:00, 826.14it/s]


,id,time,value,kind
5,"(id, 1)",0,1,kind
3,"(id, 2)",1,2,kind
1,"(id, 3)",2,3,kind
7,"(id, 4)",3,4,kind
9,"(id, 5)",4,10,kind


In [154]:
y

(id, 1)     2
(id, 2)     3
(id, 3)     4
(id, 4)    10
(id, 5)    11
Name: value, dtype: int64

### Railway efficient dataset

In [4]:
import hydra
from lmetk.data.utils import reconstruct_datetime_from_features

%load_ext autoreload
%autoreload 2
data_name = "railway_tabpfn"

with hydra.initialize(version_base="1.3", config_path="../configs", job_name="run"):
    # print('Current config file: ', config_file)
    parameters = hydra.compose(
        config_name="run.yaml",
        overrides=[f"datasource={data_name}"],
        return_hydra_config=True,
    )
    dataset_loader2 = hydra.utils.instantiate(parameters.datasource)
    dataset_loader2._pipeline()

data_splitter_params = parameters.datasource.data_splitter
seq_len = data_splitter_params.seq_len
label_len = data_splitter_params.label_len
pred_len = data_splitter_params.pred_len

# get data
splitted_data, splitted_timestamps = dataset_loader2.get_data()

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [5]:
dataset_loader2.data_dict.keys()

dict_keys(['features', 'timestamps', 'mask', 'initial_dates'])

In [6]:
parameters.datasource

{'_target_': 'lmetk.data.datasources.RailwayLoader', 'load_arguments': {'data_name': 'railway', 'data_folder': 'railway', 'data_path': '${paths.data_dir}/${datasource.load_arguments.data_folder}/${datasource.load_arguments.data_name}.csv', 'samples_per_day': 24, 'target_name': 'Bahnlast', 'timestamp_name': 'datetime', 'trend_decomp': False, 'trend_decomp_rolling_window': None, 'time_features': ['year', 'month', 'day', 'weekday', 'hour'], 'normalize_time_features': False}, 'data_splitter': {'_target_': 'lmetk.data.preprocessing.splits.DateBasedSplitter', 'test_start': '2023-01-24 00:00:00', 'train_ratio': 0.9, 'val_ratio': 0.1, 'seq_len': 24, 'label_len': 24, 'pred_len': 0, 'apply_lookback': True}, 'scaler': {'features': {'_target_': 'sklearn.preprocessing.MinMaxScaler'}}}

In [8]:
from lmetk.data.datasets.context_dataset import (
    ContextBatchDataset,
)

for mode in ["train", "val", "test"]:
    context_dataset = ContextBatchDataset(
        features=splitted_data[mode],
        time_features=splitted_timestamps[mode],
        seq_len=seq_len,
        label_len=label_len,
        pred_len=pred_len,
        mask=dataset_loader2.data_dict["mask"],
        stride=1,
    )
    print(f"ContextBatchDataset {mode} size: {context_dataset.__len__()}")
    for i in range(context_dataset.__len__()):
        context_dataset.__getitem__(i)
    d = context_dataset.__getitem__(i)

    for key in d["target"].keys():
        # pring shape of each key
        try:
            print(f"{key}: {d['target'][key].shape}")
        except:
            pass
    if mode == "test":
        # Test should stard from "2023-01-24 00:00:00"
        d = context_dataset.__getitem__(seq_len)
        print(
            reconstruct_datetime_from_features(
                d["target"]["x_mark"].long().numpy(),
                ["year", "month", "day", "weekday", "hour"],
            )[:2]
        )

ContextBatchDataset train size: 37969
x: torch.Size([24, 1])
y: torch.Size([24, 1])
x_mark: torch.Size([24, 5])
y_mark: torch.Size([24, 5])
ContextBatchDataset val size: 4225
x: torch.Size([24, 1])
y: torch.Size([24, 1])
x_mark: torch.Size([24, 5])
y_mark: torch.Size([24, 5])
ContextBatchDataset test size: 8209
x: torch.Size([24, 1])
y: torch.Size([24, 1])
x_mark: torch.Size([24, 5])
y_mark: torch.Size([24, 5])
0   2023-01-24 00:00:00
1   2023-01-24 01:00:00
dtype: datetime64[ns]


In [9]:
print((d['target']['x'].shape))
print((d['target']['y'].shape))

torch.Size([24, 1])
torch.Size([24, 1])


In [10]:
print((d['context']['x'].shape))
print((d['context']['y'].shape))

torch.Size([24, 52])
torch.Size([24, 52])


In [11]:
print((d['target']['x_mark'] == d['context']['x_mark']).all())
print((d['target']['y_mark'] == d['context']['y_mark']).all())

tensor(True)
tensor(True)


In [12]:
from lmetk.data.datamodules import ContextBaseDataModule


train_dataset = ContextBatchDataset(
        features=splitted_data['train'],
        time_features=splitted_timestamps['train'],
        seq_len=seq_len,
        label_len=label_len,
        pred_len=pred_len,
        mask=dataset_loader2.data_dict["mask"],
        stride=24,
    )


val_dataset = ContextBatchDataset(
        features=splitted_data['val'],
        time_features=splitted_timestamps['val'],
        seq_len=seq_len,
        label_len=label_len,
        pred_len=pred_len,
        mask=dataset_loader2.data_dict["mask"],
        stride=24,
    )


test_dataset = ContextBatchDataset(
        features=splitted_data['test'],
        time_features=splitted_timestamps['test'],
        seq_len=seq_len,
        label_len=label_len,
        pred_len=pred_len,
        mask=dataset_loader2.data_dict["mask"],
        stride=24,
    )

datamodule = ContextBaseDataModule(
    dataset_train=train_dataset, 
    dataset_val=val_dataset, 
    dataset_test=test_dataset,

    train_batch_size=1,
    val_batch_size=1,
    test_batch_size=1,

    shuffle_train=False,
    shuffle_val=False,
    shuffle_test=False,
    )

In [13]:
train_dataset.__len__()

1583

In [139]:
# ["year", "month", "day", "weekday", "hour"]


In [140]:
batch = next(iter(datamodule.train_dataloader()))

In [141]:
batch['target']['x_mark'].int().shape

torch.Size([1, 24, 5])

In [142]:
batch['target']['y_mark'].int().shape

torch.Size([1, 24, 5])

In [143]:
batch['target']['x_mark'].int()[:,:3]

tensor([[[2018,    4,    2,    0,    0],
         [2018,    4,    2,    0,    1],
         [2018,    4,    2,    0,    2]]], dtype=torch.int32)

In [144]:
batch['target']['y_mark'].int()[:,:3]

tensor([[[2018,    4,    2,    0,    0],
         [2018,    4,    2,    0,    1],
         [2018,    4,    2,    0,    2]]], dtype=torch.int32)

In [145]:
datamodule.train_dataloader().__len__()

37969

In [ ]:
import torch
def damodule2sklearn_naive(dl):
    X, y = [], []
    for batch in iter(dl):
        x_target, y_target, x_t_time, y_t_time = batch['target']['x'], batch['target']['y'], batch['target']['x_mark'], batch['target']['y_mark']
        x_context, y_context, x_c_time, y_c_time = batch['context']['x'], batch['context']['y'], batch['context']['x_mark'], batch['context']['y_mark']
        
        y_context, y_c_time, x_target, x_t_time

        X.append(torch.concat([x_context[:, lag, :], x_target[:, lag, :]], dim=1))
        y.append(y_t[:, lag, :])

    return torch.concat(X).numpy(), torch.concat(y).numpy()

In [39]:
dl_to_sklearn__with_context_for_tabpfn(datamodule.train_dataloader(), lag=0)

NameError: name 'y_t' is not defined

In [ ]:
x_target, y_target, x_t_time, y_t_time = batch['target']['x'], batch['target']['y'], batch['target']['x_mark'], batch['target']['y_mark']
x_context, y_context, x_c_time, y_c_time = batch['context']['x'], batch['context']['y'], batch['context']['x_mark'], batch['context']['y_mark']

concat = torch.concat([x_context, x_t_time, x_target], dim=-1)

In [87]:
concat.shape

torch.Size([1, 48, 58])

In [ ]:
from lmetk.case_studies.railway_traction import SpatialDataModule

dm = SpatialDataModule(ts_in=24, batch_size=1)
dm.setup()


In [ ]:
dm.datasets['train']

TypeError: iteration over a 0-d tensor

In [82]:
def dl_to_sklearn__with_context_for_tabpfn_original(dl, lag=0):
    X, y = [], []
    for batch in iter(dl):
        x_c, y_c, x_t, y_t = batch
        X.append(torch.concat([x_t[:, lag, :], y_c.squeeze(2)], dim=1))
        y.append(y_t[:, lag, :])

    return torch.concat(X).numpy(), torch.concat(y).numpy()

X, y = dl_to_sklearn__with_context_for_tabpfn_original(dm.train_dataloader())

In [68]:
X.shape

(39193, 80)